# Fabric Anomaly Detection with AnomalyCLIP

This notebook trains **AnomalyCLIP** (Zhou et al., ICLR 2024) on your fabric
defect dataset from Kaggle, following this pipeline:

```
Input Image -> CLIP Image Encoder -> Multi-scale Patch Features
            -> Compare with Text Prompts -> Similarity Score Per Patch
            -> Pixel Anomaly Map -> Threshold / Heatmap
            -> Defect Localization Result
```

**What AnomalyCLIP actually does under the hood** (so the diagram makes sense):
- CLIP's image encoder is run with a modified **V-V (value-value) self-attention**
  in the last few transformer blocks, which makes the patch tokens spatially
  local and suitable for *segmentation* rather than just global classification.
- Instead of hand-written prompts like `"a photo of a defect"`, AnomalyClip
  **learns** two prompt embeddings end-to-end: one that represents "normal"
  and one that represents "abnormal" (this is called *object-agnostic prompt
  learning* — it does not need the class name "fabric" at all).
- Patch tokens from **multiple transformer layers** (multi-scale) are each
  projected into CLIP's text-embedding space through small linear adapters.
- Each patch embedding is compared (cosine similarity) against the learned
  normal/abnormal text embeddings -> a per-patch anomaly score.
- Scores are reshaped into a low-res grid, upsampled to the original image
  size (bilinear interpolation) -> the pixel-level anomaly heatmap.
- A single global anomaly score for the whole image is the max (or mean-topk)
  of that map.

**Important scope note:** AnomalyCLIP is designed for *zero/few-shot* anomaly
detection — it is pretrained on auxiliary defect datasets (MVTec-AD, VisA)
and then generalizes to new objects with little/no fine-tuning. For fabric,
you have two realistic options, both supported below:

1. **Zero-shot / direct inference** — use the AnomalyCLIP checkpoint that the
   authors already trained on MVTec-AD+VisA and just run it on your fabric
   images (no training needed, works surprisingly well).
2. **Fine-tune** — continue training the prompts + adapters on your fabric
   dataset for better accuracy (recommended if you have a labelled few-shot
   set with masks or even just good/defect labels).

This notebook does **both**: it fine-tunes on your data, and shows how to
skip straight to inference if you'd rather not fine-tune.

---

### Before you run this
1. **Runtime -> Change runtime type -> GPU** (T4 or better) if running in
   Google Colab.
2. Have a **Kaggle API token** (`kaggle.json`) ready if downloading the
   dataset from Colab. If you're running this directly as a **Kaggle
   Notebook** with the dataset already attached under `/kaggle/input/...`,
   skip the download cell and just point `RAW_DATA_DIR` at that path.
3. Know your dataset's folder structure — check the "Explore dataset"
   cell output and adjust the `DATASET CONFIG` cell to match.


## 1. Environment check

In [ ]:
# Confirms a GPU is attached and shows which one.
# AnomalyCLIP's backbone is a CLIP ViT (e.g. ViT-L/14@336px) -> training
# and even inference is impractically slow on CPU.
import torch                                   # PyTorch, the deep learning framework everything else is built on
print("CUDA available:", torch.cuda.is_available())   # True/False - must be True
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))   # prints e.g. "Tesla T4"


## 2. Clone the official AnomalyCLIP repo and install dependencies

We use the authors' own implementation (`github.com/zqhang/AnomalyCLIP`) so
the V-V attention patch, prompt-learning module, and multi-scale adapter are
guaranteed to match the paper exactly — reimplementing these by hand is the
most common source of subtle bugs.


In [ ]:
# Remove any previous partial clone, then clone the official repo fresh.
!rm -rf AnomalyCLIP
!git clone https://github.com/zqhang/AnomalyCLIP.git

%cd AnomalyCLIP
# Editable install of the repo's own CLIP fork (it patches CLIP's attention
# layers, so you must use THIS version, not pip's vanilla `clip` package).
!pip install -q -e .

# Extra libraries the training/eval scripts use: image IO, metrics, progress bars.
!pip install -q opencv-python scikit-image scikit-learn tqdm ftfy regex kaggle


## 3. Download your fabric dataset from Kaggle

If you're running this notebook **inside a Kaggle Notebook** with the
dataset already attached, skip this cell entirely — your files are already
at `/kaggle/input/<dataset-name>/`.

If you're in **Colab**, run this cell. You'll be prompted to upload your
`kaggle.json` API token (get it from Kaggle -> Account -> Create New API Token).


In [ ]:
import os

RUNNING_IN_COLAB = "google.colab" in str(get_ipython())  # detects Colab vs Kaggle vs local

if RUNNING_IN_COLAB:
    from google.colab import files                # Colab's file-upload widget
    print("Upload your kaggle.json token file:")
    uploaded = files.upload()                      # opens a file picker in the browser

    os.makedirs("/root/.kaggle", exist_ok=True)    # Kaggle CLI expects the token here
    !cp kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json           # Kaggle CLI refuses to run if permissions are too open

    # ---- EDIT THIS: put your dataset's Kaggle slug here, e.g. "owner/fabric-defect-dataset"
    KAGGLE_DATASET_SLUG = "owner/your-fabric-dataset-slug"

    !kaggle datasets download -d {KAGGLE_DATASET_SLUG} -p /content/raw_data --unzip
    RAW_DATA_DIR = "/content/raw_data"
else:
    # Already on Kaggle: point this at your attached dataset folder.
    RAW_DATA_DIR = "/kaggle/input/your-fabric-dataset-slug"

print("Dataset will be read from:", RAW_DATA_DIR)


## 4. Explore the dataset structure

Run this and look at the printed tree **before** touching the config cell
below — fabric datasets on Kaggle come in very different layouts (some have
`good/` `defect/` folders, some have a CSV of labels, some have
per-defect-type subfolders like `hole/`, `stain/`, `broken_thread/`).


In [ ]:
import os

def print_tree(root, max_depth=3, max_items=15):
    # Walks the folder structure and prints a shallow tree so you can see
    # how images/labels/masks are organized, without dumping thousands of filenames.
    root = os.path.abspath(root)
    for cur_root, dirs, files in os.walk(root):
        depth = cur_root[len(root):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []   # stop descending further
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(cur_root) or cur_root}/")
        sub_indent = "  " * (depth + 1)
        for f in files[:max_items]:
            print(f"{sub_indent}{f}")
        if len(files) > max_items:
            print(f"{sub_indent}... ({len(files) - max_items} more files)")

print_tree(RAW_DATA_DIR)


## 5. Dataset config — EDIT THIS to match what you saw above

AnomalyCLIP's training/eval scripts expect **MVTec-AD-style** folders:

```
<dataset_root>/
  <class_name>/                 # e.g. "fabric"
    train/
      good/                     # only normal images, used for training
        0001.png
        ...
    test/
      good/                     # normal test images
        0001.png
      <defect_type_1>/          # e.g. "hole", "stain" ...
        0001.png
    ground_truth/
      <defect_type_1>/
        0001_mask.png           # white = defect pixels, black = background
```

Below is a converter that builds this structure from common flat layouts
(e.g. `good/` + `defect/` folders, or a labels CSV). **Adjust the mapping
logic inside `collect_samples()` to match your actual folder names from
step 4** — that's the only part of this notebook that depends on your
specific dataset.


In [ ]:
import shutil
from pathlib import Path

CLASS_NAME = "fabric"                      # AnomalyCLIP treats each object/texture type as a "class"
DATASET_ROOT = Path("/content/mvtec_format") if RUNNING_IN_COLAB else Path("/kaggle/working/mvtec_format")
HAS_MASKS = False                          # set True if your dataset provides per-pixel defect masks

def collect_samples(raw_dir):
    """
    Returns two lists of file paths: (good_images, defect_images).
    EDIT THE LOGIC BELOW to match the folder names you saw in step 4.
    This default assumes a layout like:
        raw_dir/good/*.jpg          (normal fabric)
        raw_dir/defect/*.jpg        (any defective fabric, one folder)
    """
    raw_dir = Path(raw_dir)
    good = sorted((raw_dir / "good").glob("*.*"))       # adjust folder name if needed
    defect = sorted((raw_dir / "defect").glob("*.*"))   # adjust folder name if needed
    return good, defect

good_images, defect_images = collect_samples(RAW_DATA_DIR)
print(f"Found {len(good_images)} good images, {len(defect_images)} defect images")

# --- Build MVTec-style folder tree -----------------------------------------
train_good_dir = DATASET_ROOT / CLASS_NAME / "train" / "good"
test_good_dir  = DATASET_ROOT / CLASS_NAME / "test"  / "good"
test_defect_dir = DATASET_ROOT / CLASS_NAME / "test"  / "defect"
gt_defect_dir  = DATASET_ROOT / CLASS_NAME / "ground_truth" / "defect"

for d in [train_good_dir, test_good_dir, test_defect_dir]:
    d.mkdir(parents=True, exist_ok=True)               # create folders, no error if they exist
if HAS_MASKS:
    gt_defect_dir.mkdir(parents=True, exist_ok=True)

# 90% of good images -> training (unsupervised, defect-free), 10% held out for testing.
split_idx = int(0.9 * len(good_images))
train_good, test_good = good_images[:split_idx], good_images[split_idx:]

for f in train_good:
    shutil.copy(f, train_good_dir / f.name)             # copy, keep original filename
for f in test_good:
    shutil.copy(f, test_good_dir / f.name)
for f in defect_images:
    shutil.copy(f, test_defect_dir / f.name)             # all defect images go to test (AnomalyCLIP trains only on "good")

print("Train (good):", len(list(train_good_dir.glob('*'))))
print("Test (good):", len(list(test_good_dir.glob('*'))))
print("Test (defect):", len(list(test_defect_dir.glob('*'))))


## 6. Generate the `meta.json` index file

AnomalyCLIP's data loader reads a `meta.json` describing every image's path,
label (0=normal, 1=anomaly), and mask path (if available). This cell builds
it automatically from the folder structure created above.


In [ ]:
import json

meta = {CLASS_NAME: {"train": [], "test": []}}

# ---- training split: only normal ("good") images ----
for f in sorted(train_good_dir.glob("*")):
    meta[CLASS_NAME]["train"].append({
        "img_path": f"{CLASS_NAME}/train/good/{f.name}",
        "mask_path": "",                # no mask needed for normal training images
        "cls_name": CLASS_NAME,
        "specie_name": "good",
        "anomaly": 0                    # 0 = normal
    })

# ---- test split: normal images ----
for f in sorted(test_good_dir.glob("*")):
    meta[CLASS_NAME]["test"].append({
        "img_path": f"{CLASS_NAME}/test/good/{f.name}",
        "mask_path": "",
        "cls_name": CLASS_NAME,
        "specie_name": "good",
        "anomaly": 0
    })

# ---- test split: defect images ----
for f in sorted(test_defect_dir.glob("*")):
    mask_rel = f"{CLASS_NAME}/ground_truth/defect/{f.stem}_mask.png" if HAS_MASKS else ""
    meta[CLASS_NAME]["test"].append({
        "img_path": f"{CLASS_NAME}/test/defect/{f.name}",
        "mask_path": mask_rel,          # empty string if you have no pixel-level masks
        "cls_name": CLASS_NAME,
        "specie_name": "defect",
        "anomaly": 1                    # 1 = anomalous
    })

meta_path = DATASET_ROOT / "meta.json"
with open(meta_path, "w") as fp:
    json.dump(meta, fp, indent=2)       # write the index as pretty-printed JSON

print("Wrote", meta_path)
print("Train images:", len(meta[CLASS_NAME]["train"]))
print("Test images:", len(meta[CLASS_NAME]["test"]))


## 7. Download the pretrained CLIP backbone weights

AnomalyCLIP fine-tunes prompts/adapters on top of a **frozen, pretrained
CLIP** (ViT-L/14 @ 336px by default). This downloads those base weights
(not the AnomalyCLIP-specific checkpoint yet).


In [ ]:
# open_clip provides the pretrained CLIP weights AnomalyCLIP's backbone expects.
!pip install -q open_clip_torch

import open_clip
# Downloads and caches the ViT-L-14-336 weights pretrained on OpenAI's data.
# This just warms the local cache so the training script below doesn't stall on first download.
_ = open_clip.create_model_and_transforms("ViT-L-14-336", pretrained="openai")
print("CLIP backbone weights cached.")


## 8. Fine-tune AnomalyCLIP on your fabric data

This calls the repo's own `train.py`. Only the **learnable prompts and
lightweight adapters** are trained — the CLIP backbone stays frozen, so this
is fast (a few minutes to ~1 hour on a single T4, depending on dataset size)
and needs little data.

Flag reference:
- `--train_data_path` : the MVTec-format root folder we just built
- `--dataset` : tells the script which meta.json parsing rules to use (`mvtec` layout works for our custom folder)
- `--save_path` : where trained checkpoints (prompt + adapter weights) are written
- `--features_list` : which CLIP transformer layers to pull multi-scale patch features from
- `--image_size` : input resolution fed to CLIP (336 matches the ViT-L-14-336 backbone)
- `--epoch` : training epochs — AnomalyCLIP converges fast since so few parameters are trained
- `--batch_size` : images per training step
- `--print_freq` : how often to log loss to the console


In [ ]:
!python train.py \
    --train_data_path {str(DATASET_ROOT)} \
    --dataset mvtec \
    --save_path ./checkpoints/fabric_anomalyclip \
    --features_list 6 12 18 24 \
    --image_size 336 \
    --batch_size 8 \
    --epoch 15 \
    --print_freq 10


## 9. Run inference / evaluation on the test split

`test.py` loads the checkpoint saved above, runs every test image through
the pipeline in the diagram (encode -> compare to learned prompts -> patch
similarity -> upsample to full-res heatmap), and reports metrics
(image-level AUROC always; pixel-level AUROC/AUPRO too if you supplied masks).


In [ ]:
!python test.py \
    --data_path {str(DATASET_ROOT)} \
    --dataset mvtec \
    --checkpoint_path ./checkpoints/fabric_anomalyclip/epoch_15.pth \
    --features_list 6 12 18 24 \
    --image_size 336 \
    --save_path ./results/fabric_anomalyclip


## 10. Visualize a defect localization result

Overlays the predicted pixel anomaly map on the original image, exactly as
the last step of your diagram ("Defect Localization Result").


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show_anomaly_map(image_path, anomaly_map_path, threshold=0.5):
    # Load the original fabric image (BGR -> RGB for correct display colors).
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)

    # Load the model's predicted per-pixel anomaly score map (produced by test.py,
    # saved as a single-channel image or .npy array with values in [0, 1]).
    amap = np.load(anomaly_map_path) if str(anomaly_map_path).endswith(".npy") \
        else cv2.imread(str(anomaly_map_path), cv2.IMREAD_GRAYSCALE) / 255.0

    # Binary mask of "defect" pixels: anywhere the anomaly score exceeds threshold.
    binary_mask = (amap > threshold).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img);                       axes[0].set_title("Input Image");           axes[0].axis("off")
    axes[1].imshow(amap, cmap="jet");          axes[1].set_title("Anomaly Heatmap");        axes[1].axis("off")
    axes[2].imshow(img)
    axes[2].imshow(amap, cmap="jet", alpha=0.5)   # heatmap overlaid on the original image
    axes[2].set_title(f"Localization (threshold={threshold})")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

# Example usage — update these paths to an actual test image and its saved anomaly map:
# show_anomaly_map(
#     image_path="mvtec_format/fabric/test/defect/0001.jpg",
#     anomaly_map_path="results/fabric_anomalyclip/fabric/defect/0001.npy",
#     threshold=0.5
# )


## 11. (Optional) Zero-shot inference — skip fine-tuning entirely

If you'd rather just test AnomalyCLIP's official pretrained checkpoint
(trained by the authors on MVTec-AD + VisA) directly on your fabric images
with no training at all, download their released weights and point
`--checkpoint_path` at that file instead of your fine-tuned one in step 9.
Zero-shot performance on textures like fabric is often already strong,
so it's worth trying this first as a baseline before investing time in
fine-tuning.

See the repo's README for the pretrained checkpoint download link:
https://github.com/zqhang/AnomalyCLIP


## Tips & troubleshooting

- **Out of memory**: lower `--batch_size` (try 4 or 2) or drop `--image_size` to 240.
- **No masks available**: pixel-level AUROC/AUPRO just won't be computed —
  image-level (good vs. defect) AUROC still works fine with `mask_path: ""`.
- **Multiple defect types**: instead of one `defect/` folder, keep them
  separate (`hole/`, `stain/`, `broken_thread/`) — update `collect_samples()`
  and the `meta.json` loop in step 6 to loop over each subfolder and record
  its name as `specie_name`; this gives you per-defect-type breakdowns in
  the evaluation report.
- **Very small dataset**: AnomalyCLIP only trains a small number of prompt
  and adapter parameters, so even ~50-100 "good" images can work — the
  bottleneck is usually the size of your labelled test set for evaluation,
  not the training set.
